# 🤖 End-to-End Microservice Boundary AI Agent
**System Description:** This notebook implements an AI agent that acts as an end-to-end solution for decomposing a Django monolith. It takes structural features of a Django module (Input X: imports, foreign keys, celery tasks, dependencies) and processes them through an inference pipeline to predict the target microservice cluster (Output y).


In [9]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_KEY'))

print("Доступные модели для вашего ключа:")
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

Доступные модели для вашего ключа:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview


## 🔄 Pipeline Stages
1. **Data Understanding:** Parsing the Django module's code, ORM, and framework dependencies into a feature vector (X).
2. **Reasoning:** The LLM evaluates the coupling and module semantics against standard microservice patterns.
3. **Inference & Output Generation:** The LLM assigns the module to a specific microservice cluster (y) and explains the reasoning.


In [3]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GEMINI_KEY'),
    http_options=types.HttpOptions(timeout=60000),  # 60 сек, чтобы не зависало
)

# берём из списка доступных моделей подходящую flash-модель
names = []
for m in client.models.list():
    n = m.name.replace('models/', '')
    actions = m.supported_actions or []
    if 'generateContent' in actions and 'flash' in n and not any(
        x in n for x in ['image', 'tts', 'live', 'audio', 'embedding']):
        names.append(n)

print("Доступные flash-модели:", names)
MODEL = 'gemini-3.5-flash'
print("Выбрана:", MODEL)

r = client.models.generate_content(model=MODEL, contents="Say OK")
print(r.text)
print("✅ Gemini API Client Initialized!")

Доступные flash-модели: ['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']
Выбрана: gemini-3.5-flash
OK
✅ Gemini API Client Initialized!


In [4]:
# Prompt 1: Zero-shot Example
zero_shot_prompt = """You are an AI Architect Agent specializing in decomposing Django monoliths into microservices.
Analyze the following Django module's structural features (X) and assign it to the most appropriate microservice cluster.

Input X:
Module Name: `notifications`
Features: [Imports: 15, FKs: 5, Celery Tasks: 12, DRF Serializers: 4]
Dependencies:
- Coupled to `users` (Code, ORM)
- Coupled to `payments` (Framework - Celery tasks)

Process this data, explain your reasoning briefly, and provide the final output y (Microservice Cluster Name).
"""

print("Running Zero-Shot Inference...")
response = client.models.generate_content(model=MODEL, contents=zero_shot_prompt)
print("\n=== ZERO-SHOT RESULTS ===")
print(response.text)

Running Zero-Shot Inference...

=== ZERO-SHOT RESULTS ===
### Architectural Analysis

1. **Workload Characteristics (High Async/I/O Density):**
   * The presence of **12 Celery tasks** relative to only 4 DRF Serializers indicates that the `notifications` module is primarily an asynchronous, event-driven component rather than a synchronous, request-response API.
   * Notification workloads (emails, SMS, push notifications) are typically I/O-heavy and susceptible to third-party vendor latency or rate limits.

2. **Coupling & Domain Boundaries:**
   * **`payments` Dependency:** The framework-level coupling via Celery tasks suggests `payments` already interacts with `notifications` asynchronously. This fits an **Event-Driven Architecture (Pub/Sub)** pattern, making it easy to completely decouple using message brokers (e.g., RabbitMQ, Kafka, or SQS).
   * **`users` Dependency:** The direct ORM coupling to `users` can be refactored by passing user identifiers and payload context inside event

In [5]:
# Prompt 2: Few-shot Example
few_shot_prompt = """You are an AI Architect Agent specializing in decomposing Django monoliths into microservices.
Analyze the following Django module's structural features (X) and assign it to the most appropriate microservice cluster.

Here are some examples of known mappings:

Example 1:
Input X: Module `payments`. Features: [Imports: 12, FKs: 29, Celery Tasks: 8, DRF Serializers: 15].
Output y: "Financial/Billing Service"

Example 2:
Input X: Module `homework_balancer`. Features: [Imports: 8, FKs: 14, Celery Tasks: 2, DRF Serializers: 10].
Output y: "Educational Core Service"

Now, process the following new input:
Input X:
Module Name: `gamification_rewards`
Features: [Imports: 9, FKs: 11, Celery Tasks: 5, DRF Serializers: 6]
Dependencies:
- Coupled to `homework_balancer` (Code)
- Coupled to `users` (ORM)

Process this data, explain your reasoning step-by-step, and provide the final output y.
"""

print("Running Few-Shot Inference...")
response_few_shot = client.models.generate_content(model=MODEL, contents=few_shot_prompt)
print("\n=== FEW-SHOT RESULTS ===")
print(response_few_shot.text)

Running Few-Shot Inference...

=== FEW-SHOT RESULTS ===
### Architectural Analysis & Decomposition Report

#### **Step 1: Domain & Semantic Analysis**
The module name `gamification_rewards` clearly indicates business logic centered around user incentives, achievement tracking, badges, points calculation, and reward distribution. In Domain-Driven Design (DDD), gamification represents a distinct bounded context focused on driving user retention and engagement, separate from core educational mechanics or financial transactions.

#### **Step 2: Structural & Behavioral Profile Analysis**
* **Metrics**: [Imports: 9, FKs: 11, Celery Tasks: 5, DRF Serializers: 6]
* **Async Complexity**: The high ratio of Celery tasks (5) relative to serializers (6) indicates significant asynchronous activity (e.g., calculating reward thresholds, evaluating streak milestones, updating leaderboards, and delivering background notifications).
* **Data Layer**: 11 Foreign Keys indicate moderate internal relational 

## 📝 Final Reflection
- **What the system does:** This AI agent acts as a complete end-to-end decision engine for microservice decomposition. It replaces manual heuristic-based analysis with automated semantic and structural reasoning.
- **How Gemini and prompt engineering were used:** Gemini processed the structured input (X) using zero-shot and few-shot prompting techniques. Few-shot prompting provided crucial context about the system's specific domain (EdTech/Billing), guiding the model to output accurate and consistent cluster labels (y).
- **What worked well & improvements:** The model successfully mapped graph-like node features (FKs, Celery tasks) and edges to architectural concepts. However, for a production system, providing the entire graph structure (e.g., via RAG or an expanded context window) rather than evaluating isolated nodes would improve the globally optimal boundaries.
